# Project 2: Predicting Concrete Compressive Strength

**Course:** DSC8307 Data Mining, Modelling and Analytics  
**Task:** Predict concrete compressive strength (MPa) from mixture composition and curing age.

**Data source:** `Concrete_Data.csv`, UCI Machine Learning Repository, *Concrete Compressive Strength* dataset. I-C. Yeh (1998), "Modeling of strength of high-performance concrete using artificial neural networks", *Cement and Concrete Research* 28(12), 1797–1808.

**Contents:** 1. Loading and inspecting · 2. Data preparation · 3. Mean baseline · 4. Linear vs non-linear models · 5. Hold-out test · 6. Residual analysis · 7. Sensitivity checks · 8. Association vs causation · 9. Conclusions and limitations

**To run:** put `Concrete_Data.csv` in the same folder as this notebook and install the packages in `requirements.txt`.

## 1. Loading and inspecting the data

We load the dataset, give the columns short names, and check its size, missing values and duplicates. Component quantities are in kg per m³ of mixture, age is in days, and strength is in MPa.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("Concrete_Data.csv")
df.columns = ["cement", "slag", "fly_ash", "water", "superplasticizer",
              "coarse_agg", "fine_agg", "age", "strength"]

print(df.shape)
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
df.describe().round(1)

**Observations:**
- 1030 concrete mixes, 8 inputs (7 ingredients plus age) and 1 output (strength)
- no missing values
- 25 duplicate rows
- strength ranges from about 2 to 83 MPa, with an average of about 36
- half the mixes were tested at 28 days or younger, though some were tested at up to 365 days
- many mixes contain no slag, fly ash or superplasticizer

In [ ]:
# Share of mixes that use none of each optional ingredient
(df[["slag", "fly_ash", "superplasticizer"]] == 0).mean().round(2)

### Distributions
Histograms show the shape of each variable: skew, spikes and unusual values.

In [ ]:
df.hist(bins=30, figsize=(14, 9), edgecolor="black")
plt.suptitle("Distribution of each variable", y=1.01)
plt.tight_layout()
plt.show()

**Observations:**

- **Cement** is spread fairly evenly between about 100 and 540, with no strange values.
- **Slag and fly ash** each have one tall bar at 0 (the mixes that don't use them), then a low spread of values. This matches the zero counts above.
- **Superplasticizer** looks the same: a spike at 0 and a long tail to the right (right-skewed).
- **Water, coarse aggregate and fine aggregate** are roughly bell-shaped.
- **Age** is strongly right-skewed. Most values are small, and there are only a few distinct bars, because the mixes were tested on fixed days rather than any day.
- **Strength** is roughly bell-shaped, centred around 35 MPa.

In [ ]:
# Age takes only a few fixed values (testing days chosen by the experimenters)
df["age"].value_counts().sort_index()

### Relationships with strength
The correlation heatmap shows linear associations between all variables. The scatter plots show the shape of the relationship between strength and three key inputs.

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
sns.scatterplot(data=df, x="cement", y="strength", alpha=0.5, ax=axes[0])
axes[0].set_title("Strength vs cement")
sns.scatterplot(data=df, x="age", y="strength", alpha=0.5, ax=axes[1])
axes[1].set_title("Strength vs age")
sns.scatterplot(data=df, x="water", y="strength", alpha=0.5, ax=axes[2])
axes[2].set_title("Strength vs water")
plt.tight_layout()
plt.show()

**Observations:**

**Correlations with strength:**
- cement 0.50 (strongest), superplasticizer 0.37, age 0.33, water −0.29; others weak
- inputs are related to each other: water–superplasticizer −0.66, cement–fly ash −0.40

**Scatter plots:**
- cement: upward trend but wide spread (about 10–80 MPa at 300 kg)
- age: steep rise up to about 28 days, then levels off (curved, not linear)
- water: weak downward trend, lots of scatter

## 2. Data preparation

- **Duplicates:** the 25 exact duplicate rows are removed so the same mix cannot appear in both the training and test sets, which would make test scores look better than they really are.
- **Missing values:** none, so no imputation is needed.
- **Outliers:** all values are physically plausible mix quantities, so no rows are removed.
- **Zeros in slag, fly ash and superplasticizer:** kept as real values (the ingredient is absent), not treated as missing.
- **Age:** strength rises steeply and then levels off with age, so a log(age) version is tested for the linear model in Section 4.
- **Repeated recipes:** many recipes were tested at several ages, so the same recipe can appear in both train and test at different ages. This may make scores optimistic; it is checked with a grouped cross-validation in Section 7.
- **Scaling:** applied later inside the linear model pipelines only, fitted on training data so no information leaks from the test set. Tree models do not need scaling.
- **Split:** 80% training / 20% hold-out test, with a fixed random seed for reproducibility. The test set is used only once, at the end.

In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42  # fixed seed so results are reproducible

df_clean = df.drop_duplicates().reset_index(drop=True)
print("Rows after removing duplicates:", len(df_clean))

X = df_clean.drop(columns="strength")   # inputs
y = df_clean["strength"]                # target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE)
print("Train:", X_train.shape, " Test:", X_test.shape)

## 3. Mean-prediction baseline

The baseline predicts the **training-set mean strength** for every mix. Any useful model must clearly beat it. Three metrics are reported:

- **RMSE** (root mean squared error, MPa): penalises large errors heavily
- **MAE** (mean absolute error, MPa): the typical size of an error
- **R²**: share of variation explained (0 = no better than predicting the mean)

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def evaluate(y_true, y_pred):
    """Return RMSE, MAE and R² for a set of predictions."""
    return {"RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "MAE": mean_absolute_error(y_true, y_pred),
            "R2": r2_score(y_true, y_pred)}

baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
print("Training mean strength: %.2f MPa" % y_train.mean())
pd.Series(evaluate(y_test, baseline.predict(X_test))).round(3)

**Baseline:** predicting the training mean (35.1 MPa) for every mix gives a test RMSE of about 17.3 MPa, MAE about 14.0 MPa and R² ≈ 0. Every model below must do clearly better than this.

## 4. Linear vs non-linear models

| Model | Type | Why included |
|---|---|---|
| Linear regression | Linear | Simple, interpretable reference |
| Linear regression + log(age) | Linear | Tests whether fixing the curved age effect is enough |
| Polynomial (degree 2) + Ridge | Non-linear in inputs | Captures curvature and interactions while staying a regression |
| Random forest | Non-linear (trees) | Captures interactions automatically, robust |
| Gradient boosting | Non-linear (trees) | Often the strongest on tabular data |

All models are compared using **5-fold cross-validation on the training set**, so the test set stays untouched until the end.

In [ ]:
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Replaces age with log(age), leaves the other columns unchanged
log_age = ColumnTransformer(
    [("log_age", FunctionTransformer(np.log), ["age"])],
    remainder="passthrough")

models = {
    "Baseline (mean)": DummyRegressor(strategy="mean"),
    "Linear regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Linear + log(age)": make_pipeline(log_age, StandardScaler(), LinearRegression()),
    "Polynomial (deg 2) + Ridge": make_pipeline(StandardScaler(), PolynomialFeatures(2),
                                                Ridge(alpha=1.0)),
    "Random forest": RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE),
    "Gradient boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"RMSE": "neg_root_mean_squared_error",
           "MAE": "neg_mean_absolute_error",
           "R2": "r2"}

rows = []
for name, model in models.items():
    s = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    rows.append({"Model": name,
                 "CV RMSE (mean)": -s["test_RMSE"].mean(),
                 "CV RMSE (std)": s["test_RMSE"].std(),
                 "CV MAE": -s["test_MAE"].mean(),
                 "CV R2": s["test_R2"].mean()})

cv_results = pd.DataFrame(rows).set_index("Model").round(3)
cv_results

In [ ]:
cv_results["CV RMSE (mean)"].plot(kind="barh", xerr=cv_results["CV RMSE (std)"],
                                  figsize=(8, 4), color="steelblue")
plt.xlabel("Cross-validated RMSE (MPa), lower is better")
plt.title("Model comparison (5-fold CV, training data)")
plt.gca().invert_yaxis()
plt.show()

**Observation**

Non-linear models are clearly better for this data. The tree models (about 5.3 MPa error) beat even the best linear model (about 7 MPa), and plain linear regression (about 10 MPa) does worst of all. That’s because strength depends on the inputs in curved ways (age especially) and through interactions between ingredients. Trees capture those automatically, while a linear model only can if you transform the inputs by hand, as log(age) showed.

## 5. Final evaluation on the hold-out test set

Each model is trained on the full training set and scored once on the untouched test set.

In [ ]:
fitted, test_rows = {}, []
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted[name] = model
    test_rows.append({"Model": name, **evaluate(y_test, model.predict(X_test))})

test_results = pd.DataFrame(test_rows).set_index("Model").round(3)
test_results

**Test-set results:**

- every model beats the baseline (RMSE 17.3 MPa)
- plain linear regression: RMSE 11.2 MPa, R² 0.58. Adding log(age) cuts this to 7.7 MPa (R² 0.80), showing that much of the linear model's error comes from the curved age effect
- the polynomial + Ridge model (8.2 MPa) is not better than linear + log(age)
- random forest (5.1 MPa, R² 0.91) and gradient boosting (5.5 MPa, R² 0.90) are the best, roughly halving the error of plain linear regression
- the test ranking agrees with cross-validation; random forest and gradient boosting swap places, but their difference is within the CV standard deviation (about 0.4–0.5 MPa), so they are effectively tied

## 6. Residual analysis: where are predictions weakest?

A residual is *actual − predicted*. We compare plain linear regression with gradient boosting, the non-linear model with the lowest cross-validated RMSE. We choose by cross-validation, not by the test set, so the test set stays an honest final check.

In [ ]:
compare = ["Linear regression", "Gradient boosting"]
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

for i, name in enumerate(compare):
    pred = fitted[name].predict(X_test)
    resid = y_test - pred

    ax = axes[0, i]
    ax.scatter(y_test, pred, alpha=0.6)
    lims = [0, max(y_test.max(), pred.max()) + 5]
    ax.plot(lims, lims, "r--")
    ax.set(xlabel="Actual strength (MPa)", ylabel="Predicted strength (MPa)",
           title=f"{name}: predicted vs actual")

    ax = axes[1, i]
    ax.scatter(pred, resid, alpha=0.6)
    ax.axhline(0, color="red", linestyle="--")
    ax.set(xlabel="Predicted strength (MPa)", ylabel="Residual (MPa)",
           title=f"{name}: residuals vs predicted")

plt.tight_layout()
plt.show()

**Residual observations:**

- **Linear regression:** points fan out from the diagonal. Weak mixes are over-predicted and strong mixes under-predicted, and the residual spread grows with the predicted value. This shows that two assumptions of linear regression, a straight-line relationship and constant error variance, do not hold for this data
- **Gradient boosting:** points sit close to the diagonal and residuals are mostly within ±10 MPa with no clear curve, so the non-linear model has removed most of the systematic error
- the remaining large gradient boosting errors are almost all **positive** (actual stronger than predicted) and occur for strong mixes above about 50 MPa: the model under-predicts the strongest concretes

In [ ]:
best = "Gradient boosting"
err = X_test.copy()
err["actual"] = y_test
err["abs_error"] = (y_test - fitted[best].predict(X_test)).abs()

err["age_group"] = pd.cut(err["age"], [0, 7, 28, 90, 365],
                          labels=["1-7 days", "8-28 days", "29-90 days", "91-365 days"])
err["strength_band"] = pd.cut(err["actual"], [0, 20, 40, 60, 100],
                              labels=["<20 MPa", "20-40 MPa", "40-60 MPa", ">60 MPa"])

display(err.groupby("age_group", observed=True)["abs_error"].agg(["count", "mean"]).round(2))
display(err.groupby("strength_band", observed=True)["abs_error"].agg(["count", "mean"]).round(2))

In [ ]:
# Mean residual (actual - predicted) shows systematic bias; mean absolute error shows typical error size
err["residual"] = y_test - fitted[best].predict(X_test)
err["lin_residual"] = y_test - fitted["Linear regression"].predict(X_test)
err["rel_error_%"] = 100 * err["abs_error"] / err["actual"]

display(err.groupby("strength_band", observed=True)[["lin_residual", "residual", "abs_error", "rel_error_%"]]
           .mean().round(2)
           .rename(columns={"lin_residual": "Linear: mean residual", "residual": "GB: mean residual",
                            "abs_error": "GB: MAE", "rel_error_%": "GB: relative error %"}))
print("Five largest gradient boosting errors:")
err.reindex(err["abs_error"].sort_values(ascending=False).index).head(5)[
    ["cement", "slag", "fly_ash", "water", "superplasticizer", "age", "actual", "residual"]].round(1)

**Where are predictions weakest?**

- **By strength:** errors grow with strength. For mixes above 60 MPa gradient boosting's MAE is about 8.3 MPa, against about 2.7 MPa below 20 MPa, and the mean residual is about +8 MPa, so the model systematically **under-predicts very strong concrete**. There are only 18 such mixes in the test set (and few in training), so the model has little to learn from in this region
- **In relative terms**, the weakest mixes (<20 MPa) have the largest percentage error (about 26%), because a few MPa is a large share of a small value
- **By age:** errors are largest at 8–28 days (MAE about 4.9 MPa), where strength is still rising quickly and the most varied recipes are tested
- the five largest errors are all 28-day mixes, and four of them are strong mixes (above 60 MPa) that the model under-predicted by 15–23 MPa; such high-performance recipes are rare in the data
- the linear model is biased across the whole range: about −8.5 MPa for weak mixes and about +18 MPa for mixes above 60 MPa

**Practical meaning:** the model is most trustworthy for ordinary-strength concrete (20–60 MPa). Predictions for high-strength mixes should be treated with caution and confirmed by laboratory tests.

## 7. Sensitivity checks

### Does the result depend on the particular train/test split?
We repeat the split with 10 different random seeds and record the test RMSE of the linear and gradient boosting models.

In [ ]:
from sklearn.base import clone

seed_rows = []
for seed in range(10):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=seed)
    for name in ["Linear regression", "Gradient boosting"]:
        m = clone(models[name]).fit(Xtr, ytr)   # clone = fresh copy, so the fitted models from Section 5 are not overwritten
        seed_rows.append({"seed": seed, "Model": name,
                          "RMSE": np.sqrt(mean_squared_error(yte, m.predict(Xte)))})

seeds = pd.DataFrame(seed_rows)
seeds.groupby("Model")["RMSE"].agg(["mean", "std", "min", "max"]).round(3)

**Observation:** across 10 different splits, gradient boosting's RMSE ranges from about 4.4 to 5.9 MPa and linear regression's from about 9.9 to 11.2 MPa. The ranges do not overlap, so the advantage of the non-linear model does not depend on a lucky split.

### Does repeating the same recipe at different ages inflate the scores?

Many recipes appear several times, tested at different ages. With a random split the model may have already seen the same recipe at another age. Here every recipe is kept entirely in either the training or the validation fold (GroupKFold), which is a harder and more realistic test for a *new* recipe.

In [ ]:
from sklearn.model_selection import GroupKFold

ingredients = ["cement", "slag", "fly_ash", "water", "superplasticizer", "coarse_agg", "fine_agg"]
recipe = X[ingredients].astype(str).agg("|".join, axis=1)          # one id per unique recipe
print("Unique recipes:", recipe.nunique(), " Rows:", len(X))

group_rows = []
for name in ["Linear regression", "Linear + log(age)", "Random forest", "Gradient boosting"]:
    rmse_random = -cross_validate(models[name], X, y, cv=KFold(5, shuffle=True, random_state=RANDOM_STATE),
                                  scoring="neg_root_mean_squared_error")["test_score"].mean()
    rmse_group = -cross_validate(models[name], X, y, cv=GroupKFold(5), groups=recipe,
                                 scoring="neg_root_mean_squared_error")["test_score"].mean()
    group_rows.append({"Model": name, "RMSE random folds": rmse_random, "RMSE grouped by recipe": rmse_group})
pd.DataFrame(group_rows).set_index("Model").round(2)

**Observation:**

- the 1,005 rows contain only 428 distinct recipes
- when whole recipes are held out, the tree models' error rises from about 5 MPa to about 6 MPa, so the random split was somewhat optimistic for brand-new recipes
- the linear models hardly change, and the tree models are still clearly better than the best linear model (about 7.3 MPa)
- **conclusion:** the finding that non-linear models outperform linear ones is robust, but for a genuinely new recipe a realistic error is about 6 MPa rather than 5 MPa

### Which inputs does the best model rely on?
Permutation importance measures how much the test error increases when one input is randomly shuffled.

In [ ]:
from sklearn.inspection import permutation_importance

pi = permutation_importance(fitted[best], X_test, y_test, n_repeats=20,
                            random_state=RANDOM_STATE, scoring="neg_root_mean_squared_error")
imp = pd.Series(pi.importances_mean, index=X_test.columns).sort_values()
imp.plot(kind="barh", figsize=(8, 4.5), color="darkorange")
plt.xlabel("Increase in RMSE when shuffled (MPa)")
plt.title(f"Permutation importance: {best}")
plt.show()

**Observation:** age and cement matter most (shuffling either raises the RMSE by about 10 and 9 MPa), followed by water and slag. Fly ash and coarse aggregate add little. This agrees with concrete engineering knowledge (strength develops with curing time and depends on the cement and water content), which gives some confidence the model has learned sensible patterns. Note that importance measures how much the model *uses* an input, not how much changing that input would change real strength 

## 8. Does a predictive association between cement and strength prove what happens if we change the mixture?

**Answer: No.** A predictive association between cement and strength does not prove what would happen in the real world if we changed the mixture.

1. **Observational data, not an experiment.** The dataset is a collection of different recipes. Cement was never changed on its own with everything else held fixed, so the data cannot isolate cement's own effect.
2. **The ingredients move together (confounding).** The heatmap shows cement vs fly ash at −0.40 and water vs superplasticizer at −0.66. Mixes with more cement also tend to differ in other ingredients, so part of the "cement effect" may be due to those other changes.
3. **The ingredients compete for space.** A mix fills about one cubic metre, so adding cement means less of something else (water, aggregate, slag). "More cement" is never a single change. In concrete engineering the **water-to-cement ratio** is known to be a key driver of strength (Abrams' law); in this data the water/cement ratio correlates with strength about as strongly (about −0.49) as cement alone (0.50), so the model cannot tell which one is "the" cause.
4. **Importance is not causation.** Permutation importance shows the model *relies* on cement to predict; it does not show that cement *causes* the change.
5. **Extrapolation.** Predictions are only reliable within the range of mixes in the data (e.g. cement about 102–540 kg/m³). Outside that range the model has no evidence at all.

**What would be needed:** controlled experiments that change cement while holding the other ingredients (or the water/cement ratio) fixed, together with established engineering knowledge. The model is useful for screening candidate mixes, but any change to a real mixture should be confirmed by laboratory testing.

## 9. Conclusions and limitations

**Conclusions**

- the mean baseline has an RMSE of about 17 MPa; all models beat it
- non-linear models are clearly better: random forest and gradient boosting reach about 5–5.5 MPa (R² about 0.90) against about 11 MPa for plain linear regression. Transforming age with a log closes much of the gap for the linear model (about 7.7 MPa), confirming that curvature in age is the main non-linearity
- this conclusion holds across 10 random splits and when whole recipes are held out
- predictions are weakest for high-strength concrete (above 60 MPa), which the model tends to under-predict, and for 8–28-day ages

**Limitations**

- the dataset is small (1,005 unique rows, 428 recipes), with few high-strength mixes
- for a brand-new recipe a realistic error is about 6 MPa, not 5 MPa
- hyperparameters were not tuned; tuning could reduce error slightly but is unlikely to change the comparison
- the model shows associations, not causes; it should support, not replace, laboratory testing of new mixes